In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import pandas as pd

# Define the main class for the ACO algorithm applied to bin packing
class ACOBinPacking:
    # Initialize the algorithm with item sizes, bin capacity, and parameters
    def __init__(self, item_sizes, capacity, num_ants=100, max_iter=100, alpha=0.5, beta=1.0, rho=0.1, Q=10.0, p_mut=0.8):
        self.item_sizes = np.array(item_sizes, dtype=float)
        self.n = len(item_sizes)
        self.C = float(capacity)
        if self.n < 200:
            self.num_ants = 200
            self.max_iter = 200
        elif self.n < 500:
            self.num_ants = 150
            self.max_iter = 150
        else:
            self.num_ants = num_ants
            self.max_iter = max_iter
        self.alpha = alpha # Importance of pheromone
        self.beta = beta # Importance of heuristic
        self.rho = rho # Evaporation rate
        self.Q = Q # Pheromone deposit factor
        self.p_mut = p_mut # Mutation probability
        self.p_shuffle = 0.8  # Increased for more diversity
        self.tau = np.ones((self.n, self.n)) * 0.1
        self.best_num_bins = self.n
        self.best_unused = float('inf')
        self.best_packing = None
        self.convergence = []
        self.iter_bests = []
        self.stagnation_count = 0
        self.stagnation_threshold = 5  # Lower for quicker reset
        self.exploration_phase = 0

    # Construct a solution by placing items into bins based on pheromone and heuristic information
    def construct_solution(self):
        bins = []
        bin_loads = []
        item_order = np.arange(self.n)
        if np.random.rand() < self.p_shuffle:
            np.random.shuffle(item_order)
        else:
            item_order = np.argsort(-self.item_sizes)
        alpha_temp = self.alpha
        if self.exploration_phase > 0:
            alpha_temp = 0.5
            self.exploration_phase -= 1
        for i in item_order:
            probs = []
            candidates = []
            for j in range(len(bins)):
                remaining = self.C - bin_loads[j]
                if remaining >= self.item_sizes[i]:
                    last_item = bins[j][-1] if bins[j] else -1
                    tau_val = self.tau[last_item, i] if last_item != -1 else 0.1
                    after_remaining = remaining - self.item_sizes[i]
                    after_remaining = max(1e-6, after_remaining)
                    eta = 1 / after_remaining  # Reverted to Best Fit
                    prob = (tau_val ** alpha_temp) * (eta ** self.beta)
                    # Bias towards earlier bins (lower j)
                    bias = 1.2 - (j / max(1, len(bins) - 1))
                    prob *= bias
                    probs.append(prob)
                    candidates.append(j)
            tau_new = 0.1
            after_remaining_new = self.C - self.item_sizes[i]
            after_remaining_new = max(1e-6, after_remaining_new)
            eta_new = 1 / after_remaining_new
            prob_new = (tau_new ** alpha_temp) * (eta_new ** self.beta)
            probs.append(prob_new)
            candidates.append(-1)
            selected = -1
            if candidates[:-1]:
                if np.random.rand() < 0.2:
                    selected = np.random.choice(candidates[:-1])
            if selected == -1:
                total = sum(probs)
                if total == 0:
                    probs = [1.0 / len(probs)] * len(probs)
                else:
                    probs = [p / total for p in probs]
                choice = np.random.choice(range(len(probs)), p=probs)
                selected = candidates[choice]
            if selected == -1:
                bins.append([i])
                bin_loads.append(self.item_sizes[i])
            else:
                bins[selected].append(i)
                bin_loads[selected] += self.item_sizes[i]
        return bins, bin_loads

    # Apply mutation to the solution to introduce variation
    def mutate_solution(self, packing, loads):
        if np.random.rand() >= self.p_mut:
            return packing, loads
        packing = [list(p) for p in packing]
        loads = list(loads)
        max_attempts = 10  # Increased
        for _ in range(max_attempts):
            if len(packing) < 2:
                break
            b1 = np.random.randint(len(packing))
            if not packing[b1]:
                continue
            i_idx = np.random.randint(len(packing[b1]))
            i = packing[b1][i_idx]
            size = self.item_sizes[i]
            b2 = np.random.randint(len(packing))
            while b2 == b1:
                b2 = np.random.randint(len(packing))
            if loads[b2] + size <= self.C:
                packing[b1].pop(i_idx)
                packing[b2].append(i)
                loads[b1] -= size
                loads[b2] += size
                if not packing[b1]:
                    del packing[b1]
                    del loads[b1]
        for _ in range(max_attempts):
            if len(packing) < 2:
                break
            b1 = np.random.randint(len(packing))
            b2 = np.random.randint(len(packing))
            while b2 == b1:
                b2 = np.random.randint(len(packing))
            if not packing[b1] or not packing[b2]:
                continue
            i1_idx = np.random.randint(len(packing[b1]))
            i2_idx = np.random.randint(len(packing[b2]))
            i1 = packing[b1][i1_idx]
            i2 = packing[b2][i2_idx]
            s1 = self.item_sizes[i1]
            s2 = self.item_sizes[i2]
            if loads[b1] - s1 + s2 <= self.C and loads[b2] - s2 + s1 <= self.C:
                packing[b1][i1_idx] = i2
                packing[b2][i2_idx] = i1
                loads[b1] += s2 - s1
                loads[b2] += s1 - s2
        if np.random.rand() < 0.1:
            b = np.random.randint(len(packing))
            if packing[b]:
                i_idx = np.random.randint(len(packing[b]))
                i = packing[b].pop(i_idx)
                packing.append([i])
                loads[b] -= self.item_sizes[i]
                loads.append(self.item_sizes[i])
        return packing, loads

    # Perform local search to improve the solution
    def local_search(self, packing, loads):
        packing = [list(p) for p in packing]
        loads = list(loads)
        improved = True
        passes = 0
        max_passes = 10  # Increased for better optimization
        while improved and passes < max_passes:
            improved = False
            passes += 1
            bin_indices = np.argsort(loads)
            for idx in bin_indices:
                b = idx
                if loads[b] <= 0: continue
                items = sorted(packing[b], key=lambda i: -self.item_sizes[i])  # Largest first for emptying
                temp_placements = []
                can_empty = True
                for i in items:
                    size = self.item_sizes[i]
                    candidates = []
                    for j in range(len(loads)):
                        if j == b: continue
                        if loads[j] + size <= self.C:
                            after_rem = self.C - (loads[j] + size)
                            candidates.append((after_rem + np.random.uniform(-0.1, 0.1), j))
                    if not candidates:
                        can_empty = False
                        break
                    candidates.sort(key=lambda x: x[0])
                    best_j = candidates[0][1]
                    temp_placements.append((best_j, i, size))
                if can_empty:
                    temp_loads = list(loads)
                    simulate_ok = True
                    for j, i, size in temp_placements:
                        temp_loads[j] += size
                        if temp_loads[j] > self.C + 1e-6:
                            simulate_ok = False
                            break
                    if simulate_ok:
                        for j, i, size in temp_placements:
                            packing[j].append(i)
                            loads[j] += size
                        packing[b] = []
                        loads[b] = 0
                        improved = True
                        break
            if improved:
                continue
            for b1 in range(len(packing)):
                for b2 in range(b1 + 1, len(packing)):
                    for i1 in packing[b1]:
                        for i2 in packing[b2]:
                            s1 = self.item_sizes[i1]
                            s2 = self.item_sizes[i2]
                            if loads[b1] - s1 + s2 <= self.C and loads[b2] - s2 + s1 <= self.C:
                                new_unused_b1 = self.C - (loads[b1] - s1 + s2)
                                new_unused_b2 = self.C - (loads[b2] - s2 + s1)
                                old_unused_b1 = self.C - loads[b1]
                                old_unused_b2 = self.C - loads[b2]
                                if new_unused_b1 + new_unused_b2 < old_unused_b1 + old_unused_b2:
                                    packing[b1].remove(i1)
                                    packing[b2].remove(i2)
                                    packing[b1].append(i2)
                                    packing[b2].append(i1)
                                    loads[b1] = loads[b1] - s1 + s2
                                    loads[b2] = loads[b2] - s2 + s1
                                    improved = True
                                    break
                        if improved: break
                    if improved: break
                if improved: break
            if improved:
                continue
            bin_indices_desc = np.argsort(-np.array(loads))
            bin_indices_asc = np.argsort(np.array(loads))
            for full_idx in bin_indices_desc[:5]:
                full_b = full_idx
                if loads[full_b] < self.C * 0.9: continue
                items = sorted(packing[full_b], key=lambda i: self.item_sizes[i])  # Smallest first
                for item in items:
                    size = self.item_sizes[item]
                    moved = False
                    for under_idx in bin_indices_asc[:5]:
                        under_b = under_idx
                        if under_b == full_b: continue
                        if loads[under_b] + size <= self.C:
                            packing[full_b].remove(item)
                            packing[under_b].append(item)
                            loads[full_b] -= size
                            loads[under_b] += size
                            improved = True
                            moved = True
                            break
                    if moved: break
        new_packing = [p for p in packing if p]
        new_loads = [l for l in loads if l > 0]
        return new_packing, new_loads

    # Run the ACO algorithm over multiple iterations
    def run(self):
        start_time = time.time()
        for iteration in range(self.max_iter):
            ant_solutions = []
            for ant in range(self.num_ants):
                packing, loads = self.construct_solution()
                packing, loads = self.mutate_solution(packing, loads)
                num_bins = len(packing)
                unused = sum(self.C - l for l in loads if l > 0)
                ant_solutions.append((packing, num_bins, unused, loads))
            ant_solutions.sort(key=lambda x: (x[1], x[2]))
            self.tau *= (1 - self.rho)
            for packing, num_bins, unused, loads in ant_solutions:
                delta = self.Q / num_bins
                for bin in packing:
                    for i in range(1, len(bin)):
                        prev = bin[i-1]
                        curr = bin[i]
                        self.tau[prev, curr] += delta
            packing, num_bins, unused, loads = ant_solutions[0]
            packing, loads = self.local_search(packing, loads)
            num_bins = len(packing)
            unused = sum(self.C - l for l in loads)
            self.iter_bests.append(num_bins)
            improved = False
            if num_bins < self.best_num_bins or (num_bins == self.best_num_bins and unused < self.best_unused):
                self.best_num_bins = num_bins
                self.best_unused = unused
                self.best_packing = packing
                improved = True
                self.stagnation_count = 0
            else:
                self.stagnation_count += 1
            if self.stagnation_count > self.stagnation_threshold:
                self.tau = np.ones((self.n, self.n)) * 0.1
                self.stagnation_count = 0
                self.exploration_phase = 10
                # Perturb the best solution by dispersing the bin with the most items
                if self.best_packing is not None:
                    perturbed_packing = [list(b) for b in self.best_packing]
                    perturbed_loads = [sum(self.item_sizes[k] for k in b) for b in perturbed_packing]
                    if perturbed_packing:
                        # Find bin with max number of items
                        max_items_bin = np.argmax([len(b) for b in perturbed_packing])
                        items_to_redist = perturbed_packing.pop(max_items_bin)
                        perturbed_loads.pop(max_items_bin)
                        np.random.shuffle(items_to_redist)  # Random order for redistribution
                        for item in items_to_redist:
                            placed = False
                            # Sort candidates by remaining space desc (Worst Fit for dispersion)
                            candidates = sorted(range(len(perturbed_packing)), key=lambda j: self.C - perturbed_loads[j], reverse=True)
                            for j in candidates:
                                if perturbed_loads[j] + self.item_sizes[item] <= self.C:
                                    perturbed_packing[j].append(item)
                                    perturbed_loads[j] += self.item_sizes[item]
                                    placed = True
                                    break
                            if not placed:
                                perturbed_packing.append([item])
                                perturbed_loads.append(self.item_sizes[item])
                    perturbed_num_bins = len(perturbed_packing)
                    perturbed_unused = sum(self.C - l for l in perturbed_loads)
                    # Update if better or with small probability if worse (to escape local opt)
                    if perturbed_num_bins < self.best_num_bins or (perturbed_num_bins == self.best_num_bins and perturbed_unused < self.best_unused) or np.random.rand() < 0.1:
                        self.best_num_bins = perturbed_num_bins
                        self.best_unused = perturbed_unused
                        self.best_packing = perturbed_packing
                        # Update tau based on perturbed packing
                        delta = self.Q / perturbed_num_bins
                        for bin in perturbed_packing:
                            for k in range(1, len(bin)):
                                self.tau[bin[k-1], bin[k]] += delta
            self.convergence.append(self.best_num_bins)
        runtime = time.time() - start_time
        return self.best_num_bins, self.best_unused, runtime, self.convergence

    # Plot the convergence of the algorithm
    def plot_convergence(self, filename):
        plt.figure()
        plt.plot(self.convergence, label='Best So Far')
        plt.plot(self.iter_bests, label='Iteration Best', alpha=0.7)
        plt.title('Convergence')
        plt.xlabel('Iteration')
        plt.ylabel('Best Number of Bins')
        plt.legend()
        plt.savefig(filename)
        plt.close()

    # Plot the distribution of bin loads
    def plot_load_distribution(self, filename):
        loads = [sum(self.item_sizes[k] for k in bin) for bin in self.best_packing]
        loads_sorted = sorted(loads)
        plt.figure()
        plt.bar(range(len(loads_sorted)), loads_sorted)
        plt.axhline(y=self.C, color='r', linestyle='--', label='Capacity')
        plt.title('Sorted Bin Loads')
        plt.xlabel('Bin Index (Sorted)')
        plt.ylabel('Load')
        plt.legend()
        plt.savefig(filename)
        plt.close()

# Define a subclass for 2D bin packing using area approximation
class ACO2DApproxBinPacking(ACOBinPacking):
    def __init__(self, item_dimensions, bin_dimensions, num_ants=100, max_iter=100, alpha=0.5, beta=1.0, rho=0.1, Q=10.0, p_mut=0.8):
        item_sizes = [w * h for w, h in item_dimensions]
        capacity = max([w * h for w, h in bin_dimensions])
        self.num_bins_avail = len(bin_dimensions)
        super().__init__(item_sizes, capacity, num_ants, max_iter, alpha, beta, rho, Q, p_mut)

# Function to load 1D bin packing instances from files
def load_binpack_file(filename, instance_idx=0, is_float=False):
    with open(filename, 'r') as f:
        lines = f.readlines()
    line_idx = 0
    P = int(lines[line_idx].strip())
    line_idx += 1
    for p in range(instance_idx + 1):
        identifier = lines[line_idx].strip()
        line_idx += 1
        parts = lines[line_idx].strip().split()
        capacity = float(parts[0]) if is_float else int(parts[0])
        n = int(parts[1])
        best_known = int(parts[2])
        line_idx += 1
        sizes = []
        for _ in range(n):
            if is_float:
                sizes.append(float(lines[line_idx].strip()))
            else:
                sizes.append(int(lines[line_idx].strip()))
            line_idx += 1
    return sizes, capacity, best_known, identifier

# Function to load 2D bin packing instances from files
def load_2d_binpack_file(filename, instance_idx=0):
    with open(filename, 'r') as f:
        lines = f.readlines()
    line_idx = 0
    P = int(lines[line_idx].strip())
    line_idx += 1
    for p in range(instance_idx + 1):
        while line_idx < len(lines):
            line = lines[line_idx].strip()
            if not line:
                line_idx += 1
                continue
            parts = line.split()
            if len(parts) == 2 and all(p.isdigit() for p in parts):
                line_idx += 1
                continue
            else:
                identifier = line
                line_idx += 1
                break
        else:
            raise ValueError(f"No more instances after {p-1}")
        parts = lines[line_idx].strip().split()
        n = int(parts[0])
        num_bins_avail = int(parts[1])
        if len(parts) > 2:
            best_known = int(parts[2])
        else:
            best_known = -1
        line_idx += 1
        bin_dimensions = []
        while len(bin_dimensions) < num_bins_avail:
            if line_idx >= len(lines):
                raise ValueError(f"Incomplete bin dimensions for {identifier}")
            line = lines[line_idx].strip()
            line_idx += 1
            if not line:
                continue
            parts = line.split()
            if len(parts) not in [2, 3] or not all(p.isdigit() for p in parts):
                continue
            w = int(parts[0])
            h = int(parts[1])
            count = int(parts[2]) if len(parts) == 3 else 1
            for _ in range(count):
                if len(bin_dimensions) < num_bins_avail:
                    bin_dimensions.append((w, h))
        item_dimensions = []
        skipped = 0
        for _ in range(n):
            while True:
                if line_idx >= len(lines):
                    raise ValueError(f"Incomplete items for {identifier}")
                line = lines[line_idx].strip()
                line_idx += 1
                if not line:
                    skipped += 1
                    continue
                parts = line.split()
                if len(parts) == 2 and all(p.isdigit() for p in parts):
                    item_dimensions.append((int(parts[0]), int(parts[1])))
                    break
                else:
                    skipped += 1
                    if skipped > 20:
                        raise ValueError(f"Too many invalid lines for items in {identifier}")
    return item_dimensions, bin_dimensions, best_known, identifier

# Set up directories and load 1D instances
data_dir = 'data/'
os.makedirs('plots/1D', exist_ok=True)
os.makedirs('plots/2D', exist_ok=True)
instances_1d = []
for i in range(1, 5):
    filename = os.path.join(data_dir, f'binpack{i}.txt')
    sizes, C, best_known, name = load_binpack_file(filename, 0, is_float=False)
    instances_1d.append({"name": name, "capacity": C, "best_known": best_known, "sizes": sizes})
for i in range(5, 9):
    filename = os.path.join(data_dir, f'binpack{i}.txt')
    sizes, C, best_known, name = load_binpack_file(filename, 0, is_float=True)
    instances_1d.append({"name": name, "capacity": C, "best_known": best_known, "sizes": sizes})
results_1d = []
for instance in instances_1d:
    aco = ACOBinPacking(instance["sizes"], instance["capacity"])
    num_bins, unused, runtime, convergence = aco.run()
    results_1d.append({
        "Instance": instance["name"],
        "n": len(instance["sizes"]),
        "Bin Capacity": instance["capacity"],
        "Best Known": instance["best_known"],
        "Total Used Bins": num_bins,
        "Gap": num_bins - instance["best_known"],
        "Total Unused Capacity": round(unused, 1),
        "Runtime (s)": round(runtime, 1)
    })
    aco.plot_convergence(f"plots/1D/{instance['name']}_convergence.png")
    aco.plot_load_distribution(f"plots/1D/{instance['name']}_loads.png")

# Print results for 1D instances
print("\n1D Results Table:")
df_1d = pd.DataFrame(results_1d)
print(df_1d.to_markdown(index=False, floatfmt=".1f"))

# Load and process 2D instances
instances_2d = []
filename_2d = os.path.join(data_dir, 'binpack2D.txt')
idx = 0
while True:
    try:
        item_dims, bin_dims, best_known, name = load_2d_binpack_file(filename_2d, idx)
        best_known_str = "Unknown" if best_known == -1 else best_known
        instances_2d.append({"name": name, "bin_dimensions": bin_dims, "best_known": best_known_str, "item_dimensions": item_dims})
        idx += 1
    except ValueError as e:
        if "No more instances" in str(e):
            break
        else:
            raise
results_2d = []
for instance in instances_2d:
    aco = ACO2DApproxBinPacking(instance["item_dimensions"], instance["bin_dimensions"])
    num_bins, unused, runtime, convergence = aco.run()
    if num_bins <= aco.num_bins_avail:
        aco_used = num_bins
        unused_area = round(unused, 1)
        if instance["best_known"] == "Unknown":
            gap = num_bins
        else:
            gap = num_bins - instance["best_known"]
    else:
        aco_used = "Infeasible"
        gap = "N/A"
        unused_area = "N/A"
    results_2d.append({
        "Instance": instance["name"],
        "n": len(instance["item_dimensions"]),
        "Bin Capacity": aco.C,        
        "Bins Available": len(instance["bin_dimensions"]),
        "Total Used Bins": aco_used,
        "Total Unused Capacity": unused_area,
        "Runtime (s)": round(runtime, 1)
    })
    aco.plot_convergence(f"plots/2D/{instance['name']}_convergence.png")
    aco.plot_load_distribution(f"plots/2D/{instance['name']}_loads.png")

# Print results for 2D instances
print("\n2D Results Table (Area Approximation):")
df_2d = pd.DataFrame(results_2d)
print(df_2d.to_markdown(index=False, floatfmt=".1f"))


1D Results Table:
| Instance   |    n |   Bin Capacity |   Best Known |   Total Used Bins |   Gap |   Total Unused Capacity |   Runtime (s) |
|:-----------|-----:|---------------:|-------------:|------------------:|------:|------------------------:|--------------:|
| u120_00    |  120 |          150.0 |           48 |                48 |     0 |                   122.0 |          63.4 |
| u250_00    |  250 |          150.0 |           99 |               100 |     1 |                   217.0 |          97.7 |
| u500_00    |  500 |          150.0 |          198 |               201 |     3 |                   513.0 |         130.5 |
| u1000_00   | 1000 |          150.0 |          399 |               403 |     4 |                   686.0 |         409.1 |
| t60_00     |   60 |          100.0 |           20 |                21 |     1 |                   100.0 |          26.9 |
| t120_00    |  120 |          100.0 |           40 |                43 |     3 |                   300.0 |      